# Omnilex: Creative Debate Submission
This notebook implements the creative pipeline using an **Adversarial Three-Agent Debate** with Gemini.

### Mechanism:
1. **Stage 1 & 2**: Same hybrid retrieval and graph expansion as the primary pipeline.
2. **The Debate**:
   - **Advocate**: Argues for including each of the top-30 candidates.
   - **Devil's Advocate**: Challenges the Advocate and argues for exclusion.
   - **Arbiter**: Reviews the debate and makes final binary INCLUDE/EXCLUDE decisions per citation, calibrated for F1 maximization.
3. **Hard Grounding**: Final list filtered against the corpus citation set.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Environment Detection
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    sys.path.append("/kaggle/input/omnilex-src/src")
    DATA_DIR = Path("/kaggle/input/omnilex-data/raw")
    PROCESSED_DIR = Path("/kaggle/input/omnilex-indices")
    CONFIG_DIR = Path("/kaggle/input/omnilex-config")
    # Gemini API Key should be in Kaggle Secrets
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")
else:
    PROJECT_ROOT = Path("..")
    sys.path.append(str(PROJECT_ROOT / "src"))
    DATA_DIR = PROJECT_ROOT / "data" / "raw"
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
    CONFIG_DIR = PROJECT_ROOT / "config"
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "your_api_key_here")

from omnilex.retrieval.bm25_index import BM25Index
from omnilex.retrieval.dense_retrieval import FAISSIndex, MultilingualEmbedder
from omnilex.retrieval.translator import QueryTranslator
from omnilex.retrieval.citation_graph import CitationCooccurrenceGraph
from omnilex.retrieval.reranker import CrossEncoderReranker
from omnilex.pipeline.hybrid_retriever import HybridRetriever
from omnilex.pipeline.creative_pipeline import CreativePipeline
from omnilex.evaluation.metrics import macro_f1

## 1. Configuration

In [ ]:
with open(CONFIG_DIR / "pipeline_config.json", "r") as f:
    config = json.load(f)

VAL_CSV = DATA_DIR / "val.csv"
TEST_CSV = DATA_DIR / "test.csv"

## 2. Initialize Components

In [ ]:
print("Loading indices and models...")
laws_bm25 = BM25Index.load(PROCESSED_DIR / "laws_index.pkl")
courts_bm25 = BM25Index.load(PROCESSED_DIR / "courts_index.pkl")
laws_faiss = FAISSIndex.load(PROCESSED_DIR / "laws_faiss")
courts_faiss = FAISSIndex.load(PROCESSED_DIR / "courts_faiss")
citation_graph = CitationCooccurrenceGraph.load(PROCESSED_DIR / "citation_graph.pkl")

embedder = MultilingualEmbedder(model_name=config["embedder_model"])
translator = QueryTranslator(model_name=config["translator_model"])
reranker = CrossEncoderReranker(model_name=config["reranker_model"])

laws_citations = [d["citation"] for d in laws_faiss.documents]
courts_citations = [d["citation"] for d in courts_faiss.documents]
corpus_citation_set = set(laws_citations) | set(courts_citations)

hybrid_retriever = HybridRetriever(
    laws_bm25,
    courts_bm25,
    laws_faiss,
    courts_faiss,
    embedder,
    translator,
    citation_graph,
    corpus_citation_set,
)

pipeline = CreativePipeline(
    hybrid_retriever,
    reranker,
    gemini_api_key=GEMINI_API_KEY,
    model="gemini-3-flash-preview",  # Using the latest Flash model for speed/cost
    corpus_citation_set=corpus_citation_set,
)

## 3. Creative Inference (Debate Mode)

In [ ]:
print("Running debate-based inference on test set...")
test_df = pd.read_csv(TEST_CSV)
test_queries = test_df["query"].tolist()

# Note: run_batch_with_debate includes API rate-limiting delays
test_predictions = pipeline.run_batch_with_debate(test_queries)

submission_df = pd.DataFrame(
    {
        "query_id": test_df["query_id"],
        "predicted_citations": [";".join(c) for c in test_predictions],
    }
)

submission_df.to_csv("submission_creative.csv", index=False)
print("\nCreative submission saved to submission_creative.csv")